[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabloriospena/lucirme-ai-web/blob/main/public/estudia-ia-basico/clase-3/notebook/Clase3_Basico_Practica_Guiada_EstudIA.ipynb)

# Estud-IA · IA nivel básico · Clase 3
## Práctica guiada: mira tu fuente antes de preguntarle a nadie

**Tiempo:** 20 minutos en clase. **Trabajo:** individual. **Qué necesitas:** una cuenta de Google y este notebook abierto en Colab.

### Qué vas a lograr
Al terminar habrás leído una fuente de datos con Pandas, habrás contado tres cosas y comprobado que tu conteo coincide con el del código, habrás visto qué pasa cuando la fuente cambia y habrás dejado un registro para repetir la consulta.

### Cómo se usa este notebook
1. Ejecuta cada celda con **Ctrl + Enter** (o **Mayús + Enter** para pasar a la siguiente), siempre en orden.
2. Antes de cada celda de código hay una pregunta que empieza por **Predice**. Respóndela en tu cabeza o por escrito *antes* de ejecutar. Equivocarte en la predicción es útil: te dice qué todavía no entiendes de tus datos.
3. Las celdas marcadas con **TU TURNO** necesitan que escribas algo. Las demás solo se ejecutan.
4. Si algo falla, baja a la sección **Si algo falla** al final. Casi todos los errores están ahí.

### Las tres reglas de la clase
- **Lo mínimo:** conecta solo la hoja que necesitas, no toda tu cuenta.
- **Solo lectura:** hoy leemos; no escribimos ni borramos nada en la fuente.
- **Sin datos personales:** fuentes públicas, sintéticas o con permiso.

### Mapa de este notebook
| Parte | Qué haces | Minutos |
|---|---|---|
| 0 | Preparar el entorno | 2 |
| 1 | Cargar la fuente y anotar de dónde viene | 3 |
| 2 | Mirar antes de preguntar: `head`, `shape`, `dtypes`, `isna` | 5 |
| 3 | Tres conteos y comprobación contra la hoja | 5 |
| 4 | Una pregunta que la fuente permite y otra que no | 2 |
| 5 | La fuente cambia: por qué hoy no es igual que ayer | 2 |
| 6 | Registro de la consulta | 1 |

Las partes 7 en adelante (diccionario, prueba en frío, errores, glosario y notas para quien dirige la clase) son para consulta y para el bloque de construcción.

## Parte 0 · Preparar el entorno

**Qué hace esta celda:** carga las bibliotecas que usaremos y fija la versión de lo que tienes instalado para que quede en tu registro.

- `pandas` es la biblioteca para trabajar con tablas. Se importa con el nombre corto `pd` por costumbre.
- `io` y `datetime` vienen con Python. `io` nos deja tratar un texto como si fuera un archivo y `datetime` nos da la fecha y la hora.

**Qué esperar:** una línea con la versión de Python y de Pandas. No importa cuál sea; importa que quede anotada.

In [ ]:
import io
import sys
from datetime import datetime

import pandas as pd

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

## Parte 1 · Cargar la fuente y anotar de dónde viene

### La idea
En la historia de Camila, el asistente contestó 112 el lunes y 131 el miércoles. No se equivocó: llegaron solicitudes nuevas. Lo que le faltó fue **anotar de dónde y cuándo salió el 112**. Por eso lo primero que hace este notebook, antes de mirar un solo dato, es guardar la **fuente** y la **fecha y hora** de la consulta.

### Las dos formas de entrar los datos
| | Archivo cargado | Fuente viva |
|---|---|---|
| Qué es | Una copia de tus datos en un momento | Una conexión a la fuente |
| Si la fuente cambia | La copia no cambia | El resultado puede cambiar |
| En este notebook | La **copia incluida** (viene dentro del notebook) | El **enlace CSV** de la hoja de práctica que te comparte el profesor |

### Cómo conseguir el enlace CSV de una hoja de Google
1. Abre la hoja de práctica (el enlace lo comparte el profesor en el chat).
2. Menú **Archivo → Compartir → Publicar en la Web**.
3. En la primera lista elige la pestaña de datos; en la segunda elige **Valores separados por comas (.csv)**.
4. Pulsa **Publicar** y copia el enlace. Pégalo en la variable `URL_FUENTE` de la celda siguiente.

> Un enlace publicado de esta forma es **solo lectura**: cualquiera con el enlace puede leer, nadie puede escribir. Es la forma más simple de cumplir la regla de solo lectura.

**Si no tienes el enlace todavía**, deja `URL_FUENTE` vacía. El notebook usará la copia incluida y lo anotará como **copia estática**. Para la práctica es válido, siempre que quede escrito.

### TU TURNO · pega el enlace (o déjalo vacío)

**Qué hace la celda:** define `URL_FUENTE` y la función `cargar_fuente`.
La función intenta leer el enlace. Si no hay enlace, o falla, usa la copia incluida y **te avisa en pantalla**. En los dos casos devuelve tres cosas: la tabla, el texto de origen y el tipo de fuente (`viva` o `copia`).

**Por qué devuelve el origen:** ese texto se copia luego al registro. Si la fuente es una copia, el registro lo dirá.

In [ ]:
# TU TURNO: pega aquí el enlace CSV de la hoja de práctica. Déjalo entre comillas.
# Ejemplo de forma: "https://docs.google.com/spreadsheets/d/e/....../pub?output=csv"
URL_FUENTE = ""

# --- Copia estática incluida (datos sintéticos: inventados, sin personas reales) ---
COPIA_ESTATICA = """fecha,barrio,tema,valor_reclamado,urgente
01/09/2026,Poblado,Movilidad,30000,No
01/09/2026,Belén,Servicios públicos,80000,No
01/09/2026,Belén,Seguridad,60000,No
01/09/2026,Robledo,Aseo,120000,No
01/09/2026,Laureles,Movilidad,120000,Sí
01/09/2026,Robledo,Movilidad,120000,Sí
01/09/2026,Aranjuez,Movilidad,250000,No
01/09/2026,Buenos Aires,Alumbrado,250000,No
02/09/2026,Belén,Seguridad,60000,No
02/09/2026,Belén,Vías,30000,No
02/09/2026,Poblado,Vías,0,No
02/09/2026,Castilla,,45000,No
02/09/2026,Castilla,Servicios públicos,120000,No
02/09/2026,Robledo,Movilidad,15000,Sí
02/09/2026,Robledo,Aseo,45000,No
03/09/2026,Belén,Servicios públicos,0,Sí
03/09/2026,Belén,Servicios públicos,15000,No
03/09/2026,Manrique,Servicios públicos,0,Sí
04/09/2026,Poblado,Vías,120000,Sí
04/09/2026,Manrique,Movilidad,120000,No
04/09/2026,Poblado,Servicios públicos,45000,No
04/09/2026,Robledo,Servicios públicos,120000,No
04/09/2026,Robledo,Aseo,60000,No
05/09/2026,Robledo,Alumbrado,250000,No
05/09/2026,Robledo,Alumbrado,45000,
05/09/2026,Manrique,Aseo,30000,No
05/09/2026,Robledo,Vías,45000,Sí
06/09/2026,Castilla,Alumbrado,120000,Sí
06/09/2026,Laureles,,0,Sí
06/09/2026,Poblado,Vías,15000,No
06/09/2026,Castilla,Alumbrado,15000,Sí
07/09/2026,Manrique,Servicios públicos,45000,No
07/09/2026,Aranjuez,Movilidad,15000,No
07/09/2026,Buenos Aires,,60000,No
07/09/2026,Manrique,Alumbrado,250000,Sí
07/09/2026,Aranjuez,Aseo,30000,Sí
07/09/2026,Laureles,Movilidad,0,No
07/09/2026,Castilla,Aseo,15000,Sí
07/09/2026,Belén,Vías,120000,
08/09/2026,Aranjuez,Servicios públicos,15000,No
08/09/2026,,,120000,Sí
08/09/2026,Manrique,Alumbrado,30000,No
08/09/2026,Manrique,Seguridad,80000,No
08/09/2026,Laureles,Servicios públicos,0,No
09/09/2026,Robledo,Movilidad,0,No
09/09/2026,,Aseo,30000,No
09/09/2026,Manrique,Aseo,250000,No
09/09/2026,Robledo,,30000,No
09/09/2026,Manrique,Movilidad,60000,No
09/09/2026,Laureles,,15000,No
09/09/2026,Manrique,,30000,No
10/09/2026,Robledo,Servicios públicos,45000,
10/09/2026,Laureles,Movilidad,250000,Sí
10/09/2026,Castilla,Seguridad,30000,Sí
10/09/2026,Belén,Servicios públicos,30000,No
10/09/2026,Aranjuez,Movilidad,120000,Sí
11/09/2026,Castilla,Servicios públicos,120000,No
11/09/2026,Belén,Vías,30000,Sí
12/09/2026,Buenos Aires,Seguridad,80000,No
12/09/2026,Castilla,Movilidad,80000,
13/09/2026,Castilla,Movilidad,60000,No
13/09/2026,Buenos Aires,Servicios públicos,45000,No
13/09/2026,Aranjuez,Vías,0,Sí
14/09/2026,Robledo,Seguridad,0,No
14/09/2026,Buenos Aires,Alumbrado,0,No
14/09/2026,Castilla,Alumbrado,15000,No
14/09/2026,Castilla,Alumbrado,0,No
14/09/2026,Poblado,Vías,60000,No
15/09/2026,,Servicios públicos,250000,Sí
15/09/2026,Belén,Seguridad,0,
15/09/2026,Laureles,Servicios públicos,30000,No
15/09/2026,Castilla,Vías,250000,Sí
16/09/2026,Aranjuez,Seguridad,45000,Sí
16/09/2026,Poblado,Aseo,250000,No
16/09/2026,Manrique,Vías,60000,No
16/09/2026,Aranjuez,Alumbrado,250000,Sí
16/09/2026,,Aseo,0,Sí
16/09/2026,Castilla,Seguridad,60000,Sí
16/09/2026,Belén,Servicios públicos,80000,Sí
17/09/2026,Laureles,Servicios públicos,80000,No
17/09/2026,Robledo,Vías,250000,Sí
17/09/2026,Poblado,Alumbrado,60000,Sí
17/09/2026,Robledo,Movilidad,15000,Sí
17/09/2026,Poblado,Aseo,60000,Sí
17/09/2026,Laureles,,80000,Sí
17/09/2026,Manrique,Vías,45000,No
17/09/2026,Poblado,Aseo,15000,Sí
18/09/2026,Castilla,Seguridad,60000,Sí
18/09/2026,Buenos Aires,Servicios públicos,45000,No
18/09/2026,Aranjuez,Servicios públicos,80000,No
18/09/2026,Laureles,,80000,No
18/09/2026,Laureles,Vías,45000,No
18/09/2026,,Movilidad,0,Sí
18/09/2026,Buenos Aires,Vías,80000,No
18/09/2026,Poblado,Seguridad,120000,Sí
18/09/2026,Belén,Servicios públicos,45000,No
18/09/2026,Belén,Movilidad,80000,No
19/09/2026,Aranjuez,Vías,45000,Sí
20/09/2026,Robledo,Alumbrado,45000,No
20/09/2026,Buenos Aires,Seguridad,45000,No
20/09/2026,Castilla,Seguridad,45000,Sí
20/09/2026,Laureles,Vías,30000,No
21/09/2026,,Movilidad,0,
21/09/2026,Manrique,Movilidad,80000,Sí
21/09/2026,Buenos Aires,Servicios públicos,0,No
21/09/2026,Castilla,Servicios públicos,250000,Sí
21/09/2026,Poblado,Seguridad,30000,
21/09/2026,Laureles,Movilidad,9500000,
22/09/2026,Laureles,Seguridad,15000,Sí
22/09/2026,Poblado,Alumbrado,60000,
22/09/2026,Buenos Aires,Servicios públicos,80000,
22/09/2026,Manrique,Alumbrado,0,No
22/09/2026,Castilla,,60000,No
23/09/2026,Aranjuez,Movilidad,0,
23/09/2026,Aranjuez,Servicios públicos,80000,No
23/09/2026,Manrique,Servicios públicos,0,No
23/09/2026,Belén,Vías,15000,Sí
24/09/2026,Buenos Aires,Movilidad,0,No
24/09/2026,Robledo,Movilidad,0,No
24/09/2026,Aranjuez,Servicios públicos,60000,No
24/09/2026,Buenos Aires,Seguridad,80000,No
24/09/2026,Belén,Servicios públicos,120000,No
24/09/2026,Aranjuez,Servicios públicos,60000,No
25/09/2026,Laureles,Movilidad,0,Sí
25/09/2026,Buenos Aires,Vías,250000,Sí
25/09/2026,Laureles,Servicios públicos,0,Sí
25/09/2026,Aranjuez,Seguridad,80000,Sí
25/09/2026,Robledo,Servicios públicos,30000,No
26/09/2026,Buenos Aires,Seguridad,0,No
26/09/2026,Aranjuez,Aseo,45000,Sí
26/09/2026,Buenos Aires,Seguridad,30000,Sí
26/09/2026,Aranjuez,Alumbrado,60000,No
26/09/2026,Manrique,Alumbrado,0,No
26/09/2026,Robledo,,120000,Sí
27/09/2026,Robledo,Aseo,120000,No
27/09/2026,Laureles,Movilidad,0,No
27/09/2026,Manrique,Alumbrado,120000,Sí
27/09/2026,Robledo,Aseo,60000,
27/09/2026,Castilla,Vías,0,Sí
28/09/2026,Belén,Seguridad,80000,Sí
28/09/2026,Laureles,Vías,80000,No
28/09/2026,Laureles,Movilidad,250000,No
28/09/2026,Castilla,Servicios públicos,80000,No
28/09/2026,Laureles,Servicios públicos,250000,Sí
28/09/2026,Belén,Alumbrado,250000,Sí
28/09/2026,Poblado,Vías,30000,Sí
29/09/2026,Laureles,Servicios públicos,60000,No
29/09/2026,Manrique,Servicios públicos,30000,Sí
29/09/2026,Belén,Aseo,30000,No
29/09/2026,Castilla,Aseo,80000,Sí
29/09/2026,Manrique,Alumbrado,80000,Sí
30/09/2026,Manrique,Vías,15000,No
30/09/2026,Belén,Seguridad,30000,No
30/09/2026,Robledo,Alumbrado,80000,No
30/09/2026,Castilla,Seguridad,0,No
01/10/2026,Robledo,Alumbrado,45000,Sí
01/10/2026,Laureles,,80000,Sí
01/10/2026,Robledo,Aseo,60000,Sí
01/10/2026,Buenos Aires,Aseo,15000,Sí
02/10/2026,Belén,Seguridad,0,No
02/10/2026,Poblado,Seguridad,30000,No
03/10/2026,Laureles,Alumbrado,80000,Sí
03/10/2026,Castilla,Aseo,30000,Sí
03/10/2026,,Movilidad,120000,No
03/10/2026,Poblado,,45000,No
04/10/2026,Buenos Aires,Servicios públicos,60000,No
04/10/2026,Belén,,30000,No
04/10/2026,Poblado,Servicios públicos,60000,Sí
05/10/2026,Buenos Aires,Seguridad,250000,
05/10/2026,Laureles,Alumbrado,15000,Sí
05/10/2026,Laureles,Movilidad,0,No
05/10/2026,Belén,Movilidad,80000,No
05/10/2026,,Servicios públicos,15000,Sí
05/10/2026,Laureles,,60000,No
06/10/2026,Buenos Aires,Alumbrado,80000,No
06/10/2026,Belén,Seguridad,250000,Sí
06/10/2026,,Servicios públicos,250000,No
06/10/2026,Castilla,Vías,15000,Sí
06/10/2026,Belén,Aseo,80000,Sí
07/10/2026,Buenos Aires,Seguridad,120000,Sí
07/10/2026,Aranjuez,Servicios públicos,0,Sí
07/10/2026,Robledo,Movilidad,45000,No
07/10/2026,Robledo,Seguridad,15000,No
07/10/2026,Belén,Alumbrado,80000,No
07/10/2026,Aranjuez,Aseo,0,No
07/10/2026,Robledo,Alumbrado,30000,
07/10/2026,Castilla,Seguridad,250000,No
08/10/2026,Robledo,Movilidad,0,Sí
08/10/2026,Aranjuez,Alumbrado,30000,No
08/10/2026,Robledo,Seguridad,30000,No
08/10/2026,Aranjuez,Alumbrado,120000,No
08/10/2026,Robledo,Aseo,15000,Sí
08/10/2026,,Seguridad,45000,No
09/10/2026,Castilla,Aseo,30000,Sí
09/10/2026,Poblado,Aseo,80000,No
10/10/2026,Manrique,Movilidad,30000,No
10/10/2026,Robledo,,250000,No
10/10/2026,Castilla,Vías,15000,No
10/10/2026,Laureles,Movilidad,120000,No
10/10/2026,Belén,Servicios públicos,60000,Sí
10/10/2026,Laureles,Aseo,60000,No
11/10/2026,Castilla,Vías,0,No
11/10/2026,Castilla,Movilidad,30000,No
11/10/2026,Belén,Movilidad,80000,Sí
11/10/2026,Belén,Servicios públicos,0,No
12/10/2026,Manrique,Servicios públicos,15000,No
13/10/2026,,Alumbrado,250000,Sí
13/10/2026,Castilla,Movilidad,120000,Sí
13/10/2026,Manrique,Seguridad,60000,No
14/10/2026,Laureles,Seguridad,15000,No
14/10/2026,Buenos Aires,Vías,15000,No
14/10/2026,Laureles,Servicios públicos,0,Sí
14/10/2026,Castilla,Aseo,250000,Sí
14/10/2026,Poblado,Seguridad,0,Sí
14/10/2026,Laureles,Servicios públicos,0,Sí
15/10/2026,Robledo,Seguridad,45000,
15/10/2026,Buenos Aires,Seguridad,120000,No
15/10/2026,Manrique,Vías,30000,No
16/10/2026,Aranjuez,Servicios públicos,120000,No
16/10/2026,Aranjuez,,0,No
16/10/2026,Poblado,Movilidad,60000,No
16/10/2026,Aranjuez,Vías,30000,Sí
16/10/2026,Laureles,Vías,0,No
16/10/2026,Poblado,Movilidad,45000,Sí
16/10/2026,Belén,Aseo,15000,No
16/10/2026,Laureles,Seguridad,0,Sí
17/10/2026,Belén,Servicios públicos,60000,Sí
17/10/2026,Manrique,Seguridad,120000,Sí
17/10/2026,Belén,Servicios públicos,250000,Sí
17/10/2026,Manrique,Vías,60000,No
17/10/2026,Manrique,Servicios públicos,120000,No
17/10/2026,Castilla,Movilidad,45000,Sí
18/10/2026,Castilla,Aseo,15000,No
18/10/2026,Manrique,Seguridad,30000,No
18/10/2026,Belén,Movilidad,45000,No
18/10/2026,Aranjuez,Vías,45000,No
18/10/2026,Castilla,,250000,No
18/10/2026,Buenos Aires,Alumbrado,80000,Sí
19/10/2026,Poblado,,250000,Sí
19/10/2026,Castilla,Aseo,250000,No
19/10/2026,,Alumbrado,120000,No
19/10/2026,Buenos Aires,Seguridad,250000,No
19/10/2026,Buenos Aires,,250000,Sí
19/10/2026,Manrique,Alumbrado,80000,Sí
19/10/2026,Buenos Aires,Alumbrado,0,No
19/10/2026,Aranjuez,Aseo,0,No
19/10/2026,Laureles,Vías,0,Sí
19/10/2026,Manrique,Alumbrado,0,Sí
19/10/2026,Robledo,Aseo,45000,No
19/10/2026,Manrique,Vías,120000,Sí
20/10/2026,Belén,Movilidad,250000,Sí
20/10/2026,Manrique,Vías,45000,No
20/10/2026,Buenos Aires,Seguridad,15000,Sí
20/10/2026,Poblado,Aseo,60000,Sí
21/10/2026,Belén,Movilidad,0,
21/10/2026,Castilla,Vías,60000,Sí
21/10/2026,Castilla,Aseo,250000,No
21/10/2026,Aranjuez,Alumbrado,45000,No
21/10/2026,Laureles,Vías,45000,No
21/10/2026,Buenos Aires,Vías,60000,No
21/10/2026,Castilla,Seguridad,80000,
21/10/2026,Manrique,Servicios públicos,45000,No
21/10/2026,Buenos Aires,Vías,60000,No
22/10/2026,Laureles,Aseo,80000,No
22/10/2026,Castilla,Seguridad,45000,No
22/10/2026,Poblado,Vías,0,No
23/10/2026,Buenos Aires,Movilidad,0,Sí
23/10/2026,Castilla,Movilidad,30000,Sí
24/10/2026,Aranjuez,Seguridad,250000,Sí
24/10/2026,,Aseo,0,No
24/10/2026,Castilla,Seguridad,120000,No
25/10/2026,Aranjuez,Servicios públicos,60000,No
25/10/2026,Buenos Aires,Aseo,120000,Sí
25/10/2026,Aranjuez,Seguridad,45000,No
25/10/2026,Castilla,Aseo,250000,
25/10/2026,Poblado,Alumbrado,0,No
26/10/2026,,Alumbrado,120000,Sí
26/10/2026,Manrique,,80000,Sí
26/10/2026,Poblado,Vías,30000,No
26/10/2026,Buenos Aires,Vías,80000,No
26/10/2026,Aranjuez,Alumbrado,45000,No
27/10/2026,Manrique,Servicios públicos,30000,No
27/10/2026,Belén,,80000,Sí
27/10/2026,Laureles,Movilidad,45000,No
27/10/2026,Buenos Aires,Movilidad,0,No
27/10/2026,Manrique,Vías,15000,No
28/10/2026,Laureles,Servicios públicos,15000,Sí
28/10/2026,Castilla,Aseo,80000,No
28/10/2026,Aranjuez,Movilidad,250000,No
28/10/2026,Belén,Servicios públicos,80000,Sí
29/10/2026,Laureles,Seguridad,250000,No
29/10/2026,Aranjuez,Servicios públicos,45000,Sí
29/10/2026,Buenos Aires,Servicios públicos,15000,No
30/10/2026,Laureles,Aseo,120000,No
30/10/2026,Castilla,Alumbrado,30000,Sí
30/10/2026,Aranjuez,Servicios públicos,45000,No
30/10/2026,Belén,Alumbrado,250000,No
30/10/2026,Laureles,Alumbrado,45000,Sí
31/10/2026,Poblado,Servicios públicos,30000,No
31/10/2026,Aranjuez,Vías,15000,No
"""


def cargar_fuente(url=URL_FUENTE):
    """Devuelve (tabla, origen, tipo). tipo es 'viva' o 'copia'."""
    if url.strip():
        try:
            tabla = pd.read_csv(url)
            return tabla, f"Fuente viva: {url}", "viva"
        except Exception as error:
            print("No pude leer el enlace. Uso la copia incluida.")
            print("Motivo:", type(error).__name__, "-", str(error)[:150])
    tabla = pd.read_csv(io.StringIO(COPIA_ESTATICA))
    return tabla, "Copia estática incluida en el notebook (datos sintéticos)", "copia"

### Cargar y anotar

**Qué hace la celda:** llama a `cargar_fuente()`, guarda la fecha y hora exactas de la consulta y las muestra. Esos tres datos (origen, tipo y momento) son los dos primeros del registro de la consulta.

**Qué esperar:** el origen, el tipo (`viva` o `copia`) y una fecha y hora. Ejecuta la celda dos veces: la hora cambia. Esa es la razón por la cual se anota.

In [ ]:
df, ORIGEN, TIPO_FUENTE = cargar_fuente()
MOMENTO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Origen :", ORIGEN)
print("Tipo   :", TIPO_FUENTE)
print("Momento:", MOMENTO)

## Parte 2 · Mirar antes de preguntar

Antes de pedirle algo a una IA, **mira tus datos tú**. Cuatro instrucciones de Pandas te dicen casi todo lo que necesitas saber de una tabla.

| Instrucción | Qué responde | Cómo leerla |
|---|---|---|
| `df.head()` | ¿Cómo se ven las primeras filas? | Cada fila es un registro; cada columna, un atributo |
| `df.shape` | ¿Cuántas filas y columnas hay? | Se escribe `(filas, columnas)` |
| `df.dtypes` | ¿Qué tipo de dato tiene cada columna? | `int64` es un entero, `float64` un decimal, `object` o `str` es texto |
| `df.isna()` | ¿Qué celdas están vacías? | `True` donde falta el dato; se suma para contar |

**Una nota sobre `df`:** es el nombre que le damos a la tabla (viene de *DataFrame*, que es la tabla de Pandas). Podría llamarse cualquier cosa; `df` es la costumbre.

### 2.1 · Las primeras filas con `head`

**Predice:** ¿cuántas filas va a mostrar `head()` si no le pasas ningún número? ¿Qué columnas esperas ver?

**Qué hace la celda:** muestra las primeras cinco filas. Si escribes `df.head(10)`, muestra diez.

**Qué mirar:** el nombre de cada columna y un par de ejemplos de su contenido. Fíjate en las celdas que dicen `NaN`: así muestra Pandas una celda **vacía**.

In [ ]:
df.head()

### 2.2 · El tamaño con `shape`

**Predice:** ¿cuántas filas crees que tiene la tabla? Escribe un número antes de ejecutar y compáralo con el resultado.

**Qué hace la celda:** `df.shape` devuelve dos números: filas y columnas. Las dos líneas de abajo los separan para que se lean fácil.

**Cuidado con este error frecuente:** `shape` cuenta todas las filas, incluidas las que están vacías o repetidas. Una fila repetida cuenta dos veces. Eso no es un error de Pandas; es información sobre tus datos.

In [ ]:
filas, columnas = df.shape
print("Filas   :", filas)
print("Columnas:", columnas)
print("Nombres :", list(df.columns))

### 2.3 · Los tipos con `dtypes`

**Predice:** ¿qué tipo de dato crees que tiene la columna `fecha`? ¿Y `valor_reclamado`?

**Qué hace la celda:** muestra el tipo de cada columna.

**Cómo leer el resultado:**
- `int64`: números enteros. Se pueden sumar y promediar.
- `object` o `str`: texto (según la versión de Pandas verás una u otra palabra; significan lo mismo para nosotros). Aquí también cae todo lo que Pandas no pudo tratar como número.
- La columna `fecha` aparece como `object` o `str`, es decir, **como texto**. Pandas no sabe aún que son fechas. Eso es normal en una primera lectura y se corrige en la próxima clase, cuando limpiemos y transformemos. Hoy solo lo anotamos.

In [ ]:
df.dtypes

### 2.4 · Las celdas vacías con `isna`

**Predice:** ¿en cuáles columnas crees que hay celdas vacías? ¿Cuántas en total?

**Qué hace la celda:**
- `df.isna()` devuelve una tabla del mismo tamaño con `True` donde falta el dato y `False` donde hay uno.
- `.sum()` suma los `True` de cada columna (Python cuenta `True` como 1 y `False` como 0).
- Un segundo `.sum()` suma las columnas y da el total.

**Atención, esta es la pregunta de fondo de la clase:** una celda vacía **no siempre significa lo mismo**. En `urgente`, un vacío puede querer decir «nadie la ha revisado». En `barrio`, «no lo informó». Pandas solo sabe que falta el dato; **tú** decides qué significa. Por eso en la construcción armas un diccionario de datos.

In [ ]:
vacios_por_columna = df.isna().sum()
print(vacios_por_columna)
print()
print("Total de celdas vacías:", int(vacios_por_columna.sum()))

## Parte 3 · Tres conteos comprobados contra la hoja

Un conteo del código vale lo que valga la comprobación. La regla de la clase: **cuentas tú en la hoja, cuenta el código, y se comparan**.

Los tres conteos de hoy:
1. **Filas** de la tabla.
2. **Celdas vacías** (el total).
3. **Una categoría:** cuántas filas tienen `urgente` igual a `Sí`.

### Cómo contar a mano en la hoja
- **Filas:** mira el número de la última fila y réstale 1 (la fila 1 es el encabezado).
- **Celdas vacías:** en la hoja, selecciona una columna y mira abajo a la derecha: el contador muestra cuántas celdas tienen valor. Resta ese número de las filas que esperas.
- **Categoría:** usa un filtro por columna y mira cuántas filas quedan.

> Si usas la copia estática y no tienes la hoja a mano, haz la comprobación contra otra cosa: cuenta filas con `df.shape` y comprueba con `len(df)`. No es lo ideal, pero deja claro el hábito.

### TU TURNO · escribe tus tres conteos a mano

**Qué hace la celda:** guarda lo que tú contaste. Mientras valgan `None` (vacío), la celda siguiente te lo recordará.

**Por qué escribirlos *antes* de ejecutar el código:** si ya viste la respuesta del código, ya no estás comprobando; estás copiando.

In [ ]:
# TU TURNO: reemplaza None por el número que contaste en la hoja.
MI_CONTEO_FILAS = None
MI_CONTEO_VACIOS = None
MI_CONTEO_URGENTES = None

### Los tres conteos del código y la comparación

**Qué hace la celda:**
- `len(df)` cuenta las filas.
- `df.isna().sum().sum()` suma todas las celdas vacías.
- `(df["urgente"] == "Sí").sum()` cuenta las filas donde la columna `urgente` vale exactamente `Sí`. Fíjate: el `==` compara; las filas con `NaN` o `No` no cuentan.
- La función `comprobar` compara tu número con el del código y escribe **Coincide** o **No coincide**.

**Si no coincide:** no corrijas tu número para que cuadre. Busca la causa. Las más comunes: una fila repetida, una celda con un espacio que parece vacía pero no lo está, o un filtro puesto en la hoja que oculta filas.

In [ ]:
def comprobar(nombre, mi_conteo, conteo_codigo):
    """Compara tu conteo a mano con el del código."""
    if mi_conteo is None:
        print(f"{nombre:<10} -> escribe tu conteo en la celda anterior (TU TURNO).")
        return None
    coincide = int(mi_conteo) == int(conteo_codigo)
    estado = "Coincide" if coincide else "No coincide"
    print(f"{nombre:<10} -> a mano: {mi_conteo} | código: {conteo_codigo} | {estado}")
    return coincide


CONTEO_FILAS = len(df)
CONTEO_VACIOS = int(df.isna().sum().sum())
CONTEO_URGENTES = int((df["urgente"] == "Sí").sum())

resultados = [
    comprobar("Filas", MI_CONTEO_FILAS, CONTEO_FILAS),
    comprobar("Vacías", MI_CONTEO_VACIOS, CONTEO_VACIOS),
    comprobar("Urgentes", MI_CONTEO_URGENTES, CONTEO_URGENTES),
]

### Mirar una categoría completa con `value_counts`

**Predice:** si `urgente` solo tiene `Sí` y `No`, ¿cuántos valores distintos crees que va a mostrar `value_counts`?

**Qué hace la celda:** `value_counts` cuenta cuántas veces aparece cada valor de una columna. Con `dropna=False` también cuenta los vacíos (por defecto los ignora).

**Qué mirar:** la suma de las tres líneas debe dar el total de filas. Si no suma, algo se escapó.

In [ ]:
print(df["urgente"].value_counts(dropna=False))
print()
print("Suma:", int(df["urgente"].value_counts(dropna=False).sum()), "| Filas:", len(df))

## Parte 4 · Una pregunta que la fuente permite y otra que no

Una fuente solo puede responder lo que tiene. Pedirle a una IA algo que la fuente no contiene es la forma más rápida de recibir una respuesta inventada.

**Permitida:** ¿cuántas solicitudes urgentes hay por tema? Hay una columna `tema` y una `urgente`.
**No permitida:** ¿por qué reclaman los ciudadanos? No hay ninguna columna con un motivo. El código solo puede decirte *cuántas*, nunca *por qué*.

**Qué hace la celda:** agrupa por `tema` y cuenta las filas urgentes de cada uno. Después revisa si existe una columna llamada `motivo` y te dice qué puedes afirmar.

In [ ]:
urgentes_por_tema = (
    df[df["urgente"] == "Sí"]
    .groupby("tema")
    .size()
    .sort_values(ascending=False)
)
print("Pregunta permitida: urgentes por tema")
print(urgentes_por_tema)
print()

pregunta_no_permitida = "¿Por qué reclaman los ciudadanos?"
columna_necesaria = "motivo"
print("Pregunta no permitida:", pregunta_no_permitida)
if columna_necesaria in df.columns:
    print("La fuente tiene la columna 'motivo': ya se puede intentar responder.")
else:
    print("La fuente no tiene la columna 'motivo': no puedo afirmar nada sobre las razones.")

**Mira lo que pasó con los vacíos:** las solicitudes con `tema` vacío no aparecen en la lista por tema. El conteo por tema **no suma** el total de urgentes. Eso no es un error; es lo que significa un vacío en `tema` (aún sin clasificar). Si presentaras este resultado, tendrías que decirlo.

**TU TURNO (en tu cuaderno o en un comentario de esta celda):** escribe una pregunta que *tu proyecto* permita responder y otra que no. Para cada una, di qué columna lo permite o qué columna falta.

## Parte 5 · La fuente cambia: por qué hoy no es igual que ayer

Esta parte reproduce, con datos de práctica, la historia de Camila.

**Qué hace la celda:**
1. Guarda el conteo de urgentes de ahora (`antes`).
2. Simula que llegaron solicitudes nuevas: `NUEVAS_FILAS` es un pequeño texto CSV con 25 filas fechadas en noviembre. Se unen a la tabla con `pd.concat`.
3. Vuelve a contar y muestra la diferencia.

**Predice:** si llegan 25 solicitudes nuevas, ¿el conteo de urgentes sube en 25? ¿Por qué sí o por qué no?

**Importante:** esto es una **simulación en tu copia**. No toca ninguna fuente real. En una fuente viva el cambio ocurre solo, sin que tú hagas nada.

In [ ]:
NUEVAS_FILAS = """fecha,barrio,tema,valor_reclamado,urgente
02/11/2026,Aranjuez,Alumbrado,15000,Sí
02/11/2026,Aranjuez,Aseo,30000,Sí
02/11/2026,Poblado,Vías,250000,Sí
02/11/2026,Robledo,Servicios públicos,250000,Sí
02/11/2026,Belén,Alumbrado,15000,Sí
02/11/2026,Laureles,Seguridad,250000,No
02/11/2026,Castilla,Servicios públicos,30000,Sí
02/11/2026,Poblado,Alumbrado,45000,No
02/11/2026,Castilla,Servicios públicos,45000,Sí
02/11/2026,Laureles,Movilidad,0,Sí
03/11/2026,Belén,Seguridad,80000,No
03/11/2026,Aranjuez,Alumbrado,15000,Sí
03/11/2026,Manrique,Seguridad,15000,Sí
03/11/2026,Poblado,Alumbrado,45000,Sí
03/11/2026,Robledo,Alumbrado,45000,No
03/11/2026,Laureles,Aseo,30000,Sí
03/11/2026,Poblado,Servicios públicos,45000,Sí
03/11/2026,Robledo,Aseo,80000,No
04/11/2026,Robledo,Servicios públicos,0,Sí
04/11/2026,Poblado,Movilidad,120000,No
04/11/2026,Poblado,Movilidad,15000,Sí
04/11/2026,Manrique,Movilidad,80000,Sí
04/11/2026,Robledo,Movilidad,80000,Sí
04/11/2026,Buenos Aires,Movilidad,80000,Sí
04/11/2026,Robledo,Aseo,80000,Sí
"""

antes = int((df["urgente"] == "Sí").sum())

nuevas = pd.read_csv(io.StringIO(NUEVAS_FILAS))
df_despues = pd.concat([df, nuevas], ignore_index=True)

despues = int((df_despues["urgente"] == "Sí").sum())

print("Filas antes  :", len(df), "| después:", len(df_despues))
print("Urgentes antes:", antes, "| después:", despues)
print("Diferencia    :", despues - antes)

### Qué aprendiste con esto
- La fuente cambió y la pregunta fue la misma. **La respuesta cambió y nadie se equivocó.**
- No todas las filas nuevas son urgentes: por eso el conteo no sube en 25.
- Si Camila hubiera anotado *fuente, fecha y hora*, habría sabido de inmediato que el 112 y el 131 eran fotografías de momentos distintos.

Esa es la razón del registro de la consulta, que armas ahora.

> Para volver al estado inicial, ejecuta otra vez la celda de «Cargar y anotar». `df_despues` es una variable aparte y no modifica `df`.

## Parte 6 · Registro de la consulta

El registro tiene **cuatro datos** que permiten a otra persona (o a ti en un mes) repetir la consulta:

| # | Dato | Qué va |
|---|---|---|
| 1 | Fuente | De dónde leíste |
| 2 | Fecha y hora | Cuándo hiciste la consulta |
| 3 | Instrucción | El prompt o el código que usaste |
| 4 | Comprobación | El conteo que verificaste a mano y si coincidió |

**Qué hace la celda:** arma el registro como una tabla de una fila por dato y lo muestra. Los datos 1 y 2 salen solos de la Parte 1. El 3 lo escribes tú (en este notebook, el código ya está en las celdas, así que basta con decir cuáles). El 4 se arma con los resultados de `comprobar`.

**TU TURNO:** edita `INSTRUCCION` si cambiaste algo del código.

In [ ]:
INSTRUCCION = (
    "pd.read_csv(fuente); len(df); df.isna().sum().sum(); "
    '(df["urgente"] == "Sí").sum()'
)

def texto_comprobacion(resultados):
    nombres = ["filas", "vacías", "urgentes"]
    partes = []
    for nombre, r in zip(nombres, resultados):
        estado = "sin comprobar" if r is None else ("coincide" if r else "no coincide")
        partes.append(f"{nombre}: {estado}")
    return "; ".join(partes)

registro = pd.DataFrame({
    "dato": ["1 · Fuente", "2 · Fecha y hora", "3 · Instrucción", "4 · Comprobación"],
    "valor": [ORIGEN, MOMENTO, INSTRUCCION, texto_comprobacion(resultados)],
})
pd.set_option("display.max_colwidth", None)
registro

### Guardar el registro

**Qué hace la celda:** guarda el registro como `registro_clase3.csv` en el espacio de trabajo de Colab. Ese archivo se borra cuando se cierra la sesión de Colab, así que descárgalo.

- **Colab:** la segunda parte de la celda descarga el archivo a tu computador.
- **Fuera de Colab:** guarda el archivo en la carpeta del notebook y no intenta descargar.

**Qué esperar:** una línea que dice que el archivo quedó guardado. Si ves un aviso de «No estoy en Colab», es normal fuera de Colab.

In [ ]:
registro.to_csv("registro_clase3.csv", index=False, encoding="utf-8-sig")
print("Guardé registro_clase3.csv con", len(registro), "filas.")

try:
    from google.colab import files
    files.download("registro_clase3.csv")
except ImportError:
    print("No estoy en Colab: el archivo quedó en la carpeta del notebook.")

## Parte 7 · Diccionario de datos (para el bloque de construcción)

Un **diccionario de datos** responde, para cada columna: qué significa, en qué formato viene y **qué significa un vacío**. Es lo que permite que otra persona lea tus datos sin adivinar.

**Qué hace la celda:** arma un borrador del diccionario para esta fuente de práctica. En la construcción harás el tuyo con tu propia fuente.

**Qué mirar:**
- `valor_reclamado`: un `0` es válido, significa que no hay cobro de por medio. **No** es un vacío ni un error.
- Las celdas vacías de `barrio`, `tema` y `urgente` significan cosas distintas. Tienes que escribirlo, el código no lo sabe.

In [ ]:
diccionario = pd.DataFrame([
    ["fecha",           "Día en que llegó la solicitud",   "Texto día/mes/año",  "No hay vacíos"],
    ["barrio",          "Barrio de quien solicita",        "Texto",              "Vacío: no lo informó"],
    ["tema",            "Categoría asignada",              "Texto",              "Vacío: aún sin clasificar"],
    ["valor_reclamado", "Valor reclamado, en pesos",       "Número entero",      "Un 0 es válido: no hay cobro de por medio"],
    ["urgente",         "Si la solicitud es urgente",      "Sí o No",            "Vacío: nadie la ha revisado"],
], columns=["columna", "significado", "formato", "qué significa un vacío"])
diccionario

## Parte 8 · Prueba en frío

La prueba en frío responde: **¿puedo repetir mi resultado usando solo mi registro?**

1. Reinicia el entorno: menú **Entorno de ejecución → Reiniciar sesión** (en Colab). Así se borran todas las variables.
2. Abre tu `registro_clase3.csv`. Lee solo lo que dice ahí.
3. Ejecuta de nuevo la Parte 1 y la Parte 3 siguiendo el registro, sin mirar tus notas.
4. Anota qué cambió y por qué. Hay solo dos respuestas posibles: **cambió la fuente** o **cambió la forma de preguntar**.

**Qué hace la celda siguiente:** lee el registro guardado y te muestra qué conservó. Si algo importante no está ahí, esa es tu mejora.

In [ ]:
registro_guardado = pd.read_csv("registro_clase3.csv")
registro_guardado

## Si algo falla

| Lo que ves | Qué significa | Qué hacer |
|---|---|---|
| `HTTPError: 401` o `403` al leer el enlace | La hoja no está publicada o es privada | Publícala como CSV (Parte 1) o usa la copia incluida |
| `HTTPError: 404` | El enlace está mal copiado | Cópialo de nuevo completo, sin espacios al final |
| `URLError` o `ConnectionError` | No hay conexión | Revisa tu internet; mientras tanto usa la copia incluida |
| `ParserError` | El enlace no es un CSV (suele ser la página de la hoja) | Debe terminar en `output=csv` |
| `KeyError: 'urgente'` | La columna no existe con ese nombre | Imprime `df.columns` y compara. Cuidado con mayúsculas, tildes y espacios |
| `NameError` | Una variable no existe | No ejecutaste una celda anterior. Ejecuta de arriba hacia abajo |
| Aparece `NaN` en todas partes | Es la forma de Pandas de mostrar un vacío | No es un error |
| El conteo no coincide con la hoja | Hay filas repetidas, un filtro activo en la hoja o espacios invisibles | Quita filtros, cuenta de nuevo, compara con `df.duplicated().sum()` |
| Tu uso de la IA se acabó | Plan gratuito con tope | Este notebook funciona sin IA. Anota en tu registro la hora del límite |

**Regla general:** lee el **último renglón** del error. Casi siempre dice el tipo de problema y, a veces, la columna o la línea.

## Glosario rápido

- **DataFrame (`df`):** la tabla de Pandas. Filas, columnas y un índice.
- **Fuente viva:** una conexión a los datos originales. Lo que lees puede cambiar.
- **Copia estática:** una fotografía de los datos en un momento. No cambia, y por eso hay que marcarla.
- **Celda vacía (`NaN`):** una celda sin dato. Su significado depende de la columna.
- **Registro de la consulta:** fuente, fecha y hora, instrucción y comprobación. Permite repetir.
- **Diccionario de datos:** qué significa cada columna y cada vacío.
- **Prueba en frío:** repetir tu resultado usando solo el registro.
- **Solo lectura:** acceso que permite leer y no cambiar.
- **Conector:** conexión entre una herramienta de IA y una de tus fuentes.

## Antes de pasar al notebook de tu proyecto
- [ ] Mis tres conteos coinciden con la hoja (o sé por qué no).
- [ ] Mi registro tiene los cuatro datos.
- [ ] Sé qué pregunta permite mi fuente y cuál no.
- [ ] Entiendo por qué un vacío no siempre significa lo mismo.

## Notas para quien dirige la clase

**Solo para el profesor.** Valores de la copia estática incluida (sirven para verificar que todo corre y para guiar sin sorpresas):

| Concepto | Valor esperado |
|---|---|
| Filas | 300 |
| Columnas | 5 (`fecha`, `barrio`, `tema`, `valor_reclamado`, `urgente`) |
| Celdas vacías | 54 (`barrio` 14, `tema` 22, `urgente` 18) |
| Urgentes (`Sí`) | 112 |
| No urgentes (`No`) | 170 |
| Filas repetidas exactas | 3 (se tratan en la próxima clase) |
| Valores con 0 en `valor_reclamado` | 51 (válidos) |
| Valor atípico | 1 fila con 9.500.000 (visible, no imposible) |
| Tras sumar las 25 nuevas | 325 filas; urgentes 131 |

**Hilo con la clase:** el 112 → 131 reproduce el caso de Camila de la diapositiva de apertura.
**La hoja de práctica:** hay que crearla en Google Sheets con estos mismos datos (se puede importar `registro` desde el CSV de la copia) y publicarla como CSV. Hasta entonces, el notebook funciona con la copia incluida.
**La celda de abajo** comprueba que la copia está intacta. Ejecútala solo si `TIPO_FUENTE == "copia"`.

In [ ]:
if TIPO_FUENTE == "copia":
    assert df.shape == (300, 5), df.shape
    assert int(df.isna().sum().sum()) == 54
    assert int((df["urgente"] == "Sí").sum()) == 112
    assert int((df["urgente"] == "No").sum()) == 170
    assert int(df.duplicated().sum()) == 3
    assert int((df["valor_reclamado"] == 0).sum()) == 51
    assert despues == 131 and len(df_despues) == 325
    print("Autoprueba superada: la copia estática está intacta.")
else:
    print("La fuente es viva: los valores esperados no aplican.")